# Part A - Data Prep PipelineAll data-prep steps in one notebook, run top to bottom:1. Download data (local only - skip on Kaggle, data is already attached)2. Convert DICOM to YOLO format3. Train/val split4. Create dataset.yamlEach cell defaults to LOCAL paths. If running on Kaggle, uncomment theKAGGLE path block in each cell and comment out the LOCAL block above it.

## Setup

In [ ]:
from pathlib import Pathimport osimport randomimport shutilimport subprocessimport zipfileimport cv2import pydicomimport pandas as pdfrom tqdm import tqdm# this notebook lives in part_a_pneumonia_detection/, so this is the base folderBASE_DIR = Path.cwd()print("base dir:", BASE_DIR)

## Step 1: Download data (LOCAL ONLY)Skip this cell entirely if running on Kaggle - the dataset is alreadyattached to the notebook automatically.One-time setup before running this locally:1. `pip install kaggle`2. kaggle.com -> profile -> Settings -> API -> "Create New Token" (downloads kaggle.json)3. Put kaggle.json at `~/.kaggle/kaggle.json` (Mac/Linux) or `C:\Users\<you>\.kaggle\kaggle.json` (Windows)4. Join the competition once at kaggle.com/c/rsna-pneumonia-detection-challenge

In [ ]:
DATA_DIR = BASE_DIR / "data"ZIP_PATH = DATA_DIR / "rsna-pneumonia-detection-challenge.zip"COMPETITION = "rsna-pneumonia-detection-challenge"DATA_DIR.mkdir(exist_ok=True)print("downloading dataset (this is a few GB, may take a while)...")subprocess.run(    ["kaggle", "competitions", "download", "-c", COMPETITION, "-p", str(DATA_DIR)],    check=True,)print("unzipping...")with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:    zip_ref.extractall(DATA_DIR)print("done. data is in:", DATA_DIR)

## Step 2: Convert DICOM to YOLO format

In [ ]:
# ---- LOCAL paths (default) ----DICOM_DIR = BASE_DIR / "data" / "stage_2_train_images"LABELS_CSV = BASE_DIR / "data" / "stage_2_train_labels.csv"OUT_IMG_DIR = BASE_DIR / "dataset" / "images" / "train"OUT_LBL_DIR = BASE_DIR / "dataset" / "labels" / "train"# ---- KAGGLE paths (uncomment these, comment out the LOCAL block above, if running on Kaggle) ----# DICOM_DIR = Path("/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images/")# LABELS_CSV = Path("/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv")# OUT_IMG_DIR = Path("/kaggle/working/dataset/images/train/")# OUT_LBL_DIR = Path("/kaggle/working/dataset/labels/train/")# RSNA images are always 1024x1024 - needed to normalize the box coordinatesIMG_SIZE = 1024# how many images to convert - full dataset is ~25000, use a smaller# subset first to keep things fast while iteratingNUM_IMAGES = 2000OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)OUT_LBL_DIR.mkdir(parents=True, exist_ok=True)labels = pd.read_csv(LABELS_CSV)patient_ids = labels["patientId"].unique()[:NUM_IMAGES]for pid in tqdm(patient_ids, desc="converting DICOM to YOLO format"):    dcm_path = DICOM_DIR / f"{pid}.dcm"    if not dcm_path.exists():        continue    dicom_file = pydicom.dcmread(dcm_path)    image = dicom_file.pixel_array    cv2.imwrite(str(OUT_IMG_DIR / f"{pid}.png"), image)    patient_rows = labels[labels["patientId"] == pid]    label_lines = []    for _, row in patient_rows.iterrows():        if row["Target"] == 1:            x, y, w, h = row["x"], row["y"], row["width"], row["height"]            # YOLO wants center-x, center-y, width, height, normalized 0-1            x_center = (x + w / 2) / IMG_SIZE            y_center = (y + h / 2) / IMG_SIZE            w_norm = w / IMG_SIZE            h_norm = h / IMG_SIZE            # class 0 = pneumonia (only one class in this problem)            label_lines.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")    # empty label file if no pneumonia - YOLO needs to see "no object" examples too    label_path = OUT_LBL_DIR / f"{pid}.txt"    with open(label_path, "w") as f:        f.write("\n".join(label_lines))print(f"done. converted {len(patient_ids)} images")

## Step 3: Train/val split

In [ ]:
# ---- LOCAL paths (default) ----TRAIN_IMG_DIR = BASE_DIR / "dataset" / "images" / "train"TRAIN_LBL_DIR = BASE_DIR / "dataset" / "labels" / "train"VAL_IMG_DIR = BASE_DIR / "dataset" / "images" / "val"VAL_LBL_DIR = BASE_DIR / "dataset" / "labels" / "val"# ---- KAGGLE paths (uncomment these, comment out the LOCAL block above, if running on Kaggle) ----# TRAIN_IMG_DIR = Path("/kaggle/working/dataset/images/train/")# TRAIN_LBL_DIR = Path("/kaggle/working/dataset/labels/train/")# VAL_IMG_DIR = Path("/kaggle/working/dataset/images/val/")# VAL_LBL_DIR = Path("/kaggle/working/dataset/labels/val/")VAL_FRACTION = 0.15  # 15% of images go to validation, 85% stay for trainingVAL_IMG_DIR.mkdir(parents=True, exist_ok=True)VAL_LBL_DIR.mkdir(parents=True, exist_ok=True)all_images = list(TRAIN_IMG_DIR.iterdir())# fixed seed so the split is reproducible run to runrandom.seed(42)random.shuffle(all_images)val_count = int(VAL_FRACTION * len(all_images))val_images = all_images[:val_count]for img_path in val_images:    base_name = img_path.stem    shutil.move(str(img_path), str(VAL_IMG_DIR / img_path.name))    shutil.move(str(TRAIN_LBL_DIR / f"{base_name}.txt"), str(VAL_LBL_DIR / f"{base_name}.txt"))print("train:", len(list(TRAIN_IMG_DIR.iterdir())))print("val:", len(list(VAL_IMG_DIR.iterdir())))

## Step 4: Create dataset.yamlNote: on Kaggle, `/kaggle/working/` gets wiped every session, so thiscell needs to be re-run each session even if run before. Locally, thisfile just sits in the project folder and only needs regenerating if thedata location changes.

In [ ]:
# ---- LOCAL path (default) ----OUTPUT_PATH = BASE_DIR / "dataset.yaml"DATASET_ROOT = BASE_DIR / "dataset"# ---- KAGGLE path (uncomment this, comment out the LOCAL block above, if running on Kaggle) ----# OUTPUT_PATH = Path("/kaggle/working/dataset.yaml")# DATASET_ROOT = Path("/kaggle/working/dataset")yaml_content = f"""\path: {DATASET_ROOT}train: images/trainval: images/valnc: 1names: ["pneumonia"]"""with open(OUTPUT_PATH, "w") as f:    f.write(yaml_content)print("created:", OUTPUT_PATH)

Data prep done. Next: open `train.py` (or run its cells if you also convert that to a notebook) to train the model.